In [12]:
import pandas as pd
import pyarrow.parquet as pq
import fsspec
import time
import random

years = range(2023, 2026)
months = range(1, 2)
base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month:02d}.parquet"

frames = []

# Mantenemos el bloque grande y el User-Agent para optimizar la red
fs = fsspec.filesystem(
    'https', 
    block_size=5 * 1024 * 1024, 
    client_kwargs={'headers': {'User-Agent': 'Mozilla/5.0 (Data Analytics Pipeline)'}}
)

for year in years:
    for month in months:
        url = base_url.format(year=year, month=month)
        
        # Añadimos un retraso aleatorio ("politeness delay") entre 1 y 3 segundos
        # Esto evita activar las alarmas de Rate Limiting del CDN
        time.sleep(random.uniform(5.0, 10.0))
        
        try:
            with fs.open(url, 'rb') as f:
                pf = pq.ParquetFile(f)
                batches = pf.iter_batches(batch_size=1000)
                df_batch = next(batches).to_pandas()
                
                df_batch['source_year'] = year
                df_batch['source_month'] = month
                
                # Solución al FutureWarning: 
                # Eliminamos columnas que sean 100% nulas en este lote antes de concatenar.
                # Esto alinea mejor los esquemas y evita la advertencia de Pandas.
                df_batch = df_batch.dropna(axis=1, how='all')
                
                frames.append(df_batch)
                print(f"Éxito: {year}-{month:02d}")
                
        except Exception as e:
            print(f"Error procesando {year}-{month:02d}: {type(e).__name__} - {str(e)[:50]}")
            continue

if frames:
    # Concatenamos los dataframes limpios
    df_final = pd.concat(frames, ignore_index=True)
    print(f"\nExtracción completada. Shape final: {df_final.shape}")
    
    # Práctica Analítica: Validamos cuántas columnas sobrevivieron a la deriva de esquemas
    print(f"Columnas resultantes: {len(df_final.columns)}")
else:
    print("No se encontraron datos.")


Error procesando 2023-01: FileNotFoundError - https://d37ci6vzurychx.cloudfront.net/trip-data/ye
Error procesando 2024-01: FileNotFoundError - https://d37ci6vzurychx.cloudfront.net/trip-data/ye
Error procesando 2025-01: FileNotFoundError - https://d37ci6vzurychx.cloudfront.net/trip-data/ye
No se encontraron datos.


In [6]:
df_final.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'airport_fee', 'source_year',
       'source_month', 'Airport_fee', 'cbd_congestion_fee'],
      dtype='object')

In [3]:
df_final.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'airport_fee', 'source_year',
       'source_month'],
      dtype='object')

In [4]:
years = range(2023, 2026) 
months = range(1, 2)
batch_size = 1000

base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month:02d}.parquet"

frames = []
fs = fsspec.filesystem('https') # Permite lecturas parciales (Range requests) sobre HTTP

for year in years:
    for month in months:
        url = base_url.format(year=year, month=month)
        
        try:
            # Abrimos la conexión sin descargar todo el archivo
            with fs.open(url, 'rb') as f:
                # Leemos solo la metadata (footer) del parquet
                pf = pq.ParquetFile(f)
                
                # iter_batches extrae solo las filas necesarias usando el tamaño de lote definido
                batches = pf.iter_batches(batch_size=batch_size)
                
                # Extraemos el primer lote y lo pasamos a pandas
                df_batch = next(batches).to_pandas()
                
                # Añadimos variables de partición (buena práctica analítica)
                df_batch['source_year'] = year
                df_batch['source_month'] = month
                
                frames.append(df_batch)
                
        except FileNotFoundError:
            # fsspec levanta FileNotFoundError para 403/404 en este contexto
            print(f"Archivo no encontrado o no disponible: {year}-{month:02d}")
            continue
        except Exception as e:
            print(f"Error inesperado procesando {year}-{month:02d}: {str(e)}")
            continue

if frames:
    df_final1 = pd.concat(frames, ignore_index=True)
    print(f"Extracción completada. Shape: {df_final1.shape}")
else:
    print("No se encontraron datos.")


Archivo no encontrado o no disponible: 2023-01
Archivo no encontrado o no disponible: 2024-01
Archivo no encontrado o no disponible: 2025-01
No se encontraron datos.
